# 초기 100 사이클 피처 재현

초기 피처 수식을 원본 MAT에서 재계산하고 모델 입력 및 메타데이터를 분리한다.

In [1]:
from pathlib import Path
import os, sys, json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display, Image
# 프로젝트 루트는 현재 폴더 또는 부모 폴더에서 찾습니다.
project_override = os.environ.get('ESS_PROJECT_DIR') or os.environ.get('ESS_DAY2_PROJECT')
project_choices = ([Path(project_override).expanduser()] if project_override else
                   [Path.cwd(), Path.cwd() / 'day2_project',
                    Path.cwd().parent, Path.cwd().parent / 'day2_project'])
PROJECT_DIR = next((p.resolve() for p in project_choices
                    if (p / 'src/train.py').is_file()
                    and (p / 'data/processed/cells_and_features.csv').is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('프로젝트 루트에서 실행하거나 ESS_PROJECT_DIR로 프로젝트 폴더를 지정하세요.')
MODEL_PROJECT_DIR = PROJECT_DIR
if str(MODEL_PROJECT_DIR) not in sys.path: sys.path.insert(0,str(MODEL_PROJECT_DIR))
DAY2_OUT=Path(os.environ.get('ESS_DAY2_OUT',str(MODEL_PROJECT_DIR/'results')))
DAY2_OUT.mkdir(parents=True,exist_ok=True)
from src.preprocess import FEATURE_SETS, validate_features, split_development, make_model, metrics
from src.train import PROTOCOL, candidate_specs, dump_json, compare_candidates, evaluate_locked
from src.visualize import build_charts
from src.diagnostics import diagnose
from src.quality import quality_sensitivity


In [2]:
from src.features import extract_features
from src.quality import author_rules
# 기본 원본 폴더는 data/raw이며, 기존 실습의 archive 폴더도 인식합니다.
raw_choices = [PROJECT_DIR / 'data/raw', PROJECT_DIR / 'archive', PROJECT_DIR.parent / 'archive']
default_raw_dir = next((p for p in raw_choices
                        if (p / '2017-05-12_batchdata_updated_struct_errorcorrect.mat').is_file()),
                       PROJECT_DIR / 'data/raw')
DATA_DIR = Path(os.environ.get('ESS_DATA_DIR', str(default_raw_dir))).expanduser().resolve()
features,source_manifest,current_quality,current_rejected=extract_features(DATA_DIR)
processed=MODEL_PROJECT_DIR/'data/processed';processed.mkdir(parents=True,exist_ok=True)
features.to_csv(processed/'cells_and_features.csv',index=False)
source_manifest.to_csv(processed/'source_manifest.csv',index=False)
current_quality.to_csv(processed/'current_quality.csv',index=False)
current_rejected.to_csv(processed/'current_rejected.csv',index=False)
author_rules(DATA_DIR).to_csv(processed/'batch3_author_quality_rules.csv',index=False)
display(features.head(3));display(current_quality)

Batch 1: 46셀, 38,811사이클 행 로딩 완료
Batch 2: 47셀, 26,904사이클 행 로딩 완료
Batch 3: 46셀, 51,007사이클 행 로딩 완료


,cell_id,batch,cell_index,cycle_life,charging_policy,summary_cycles,QD_bad_rows,tail_median_QD,eol_near,curve_valid,...,fast_charge_duration_min,current_cycle_n,current_unit_ratio_A_per_C,charge_integral_ratio,QD_cycle2,mean_IR,delta_IR,mean_Tavg,mean_Tmax,mean_chargetime
0,b1c0,1,0,1190.0,3.6C(80%)-3.6C,1189,2,1.026499,False,True,...,13.342660,99,1.100541,0.999974,1.070689,0.016643,0.000037,31.603747,35.376748,13.384231
1,b1c1,1,1,1179.0,3.6C(80%)-3.6C,1178,1,1.038700,False,True,...,13.340656,99,1.100178,1.000007,1.075301,0.016922,0.000060,31.330314,34.160655,13.374696
2,b1c2,1,2,1177.0,3.6C(80%)-3.6C,1176,1,1.043586,False,True,...,13.340589,99,1.100178,1.000003,1.079922,0.016769,0.000095,31.479584,34.544644,13.378886


,batch,cell_n,cycle_n,min_cycle,max_cycle,median_unit_ratio,median_charge_ratio,min_charge_ratio,max_charge_ratio,zero_time_intervals
0,1,46,4551,2,100,1.100190,0.999807,0.993442,1.004983,1
1,2,47,4634,2,100,1.100360,0.999973,0.996266,1.001648,3
2,3,46,4554,2,100,1.100399,0.999943,0.998863,1.008113,15


## 초기 입력 수식

logvar_deltaQ=log10(sample var(Q100-Q10)), slope_QD=cycle2~100 선형 기울기, delta_IR=91~100 평균-2~10 평균, 온도·충전시간=2~100 유효 양수 평균. 전류 RMS/CV는 공칭 0.88Ah까지 시간 가중 통계를 구한 뒤 초기 사이클 평균을 사용한다. Qdlin은 셀별 실제 Vdlin 축을 정렬하고 동일 셀의 두 곡선을 차분한다.

In [3]:
display(pd.DataFrame([{'feature_set':k,'features':', '.join(v)} for k,v in FEATURE_SETS.items()]))
reference=pd.read_csv(processed/'eda_reference_features.csv')
pd.testing.assert_frame_equal(features,reference,check_exact=False,rtol=1e-10,atol=1e-12)
assert current_quality.max_cycle.max()==100
print('139×31 DAY 1 피처 표와 재추출값 일치')

,feature_set,features
0,single,logvar_deltaQ
1,physical,"logvar_deltaQ, slope_QD, delta_IR, mean_Tavg, ..."
2,policy,"logvar_deltaQ, slope_QD, delta_IR, mean_Tavg, ..."
3,current,"logvar_deltaQ, slope_QD, delta_IR, mean_Tavg, ..."


139×31 DAY 1 피처 표와 재추출값 일치
